### CESM-HR 전처리 — 월평균 SST 서브셋\n\n
1. 전체 기간(year 250–349) 월평균 파일 로드
2. 북서태평양 도메인 서브셋 3종\n   - **a** : lat 10–55°N, lon 105–180°E\n   - **b** : lat 15–45°N, lon 120–180°E\n   - **c** : lat 20–35°N, lon 125–150°E (core)
3. 도메인별 NetCDF 저장
- CESM-HR POP 해양모델은 **삼중극(tripolar) 곡선격자** (nlat=2400, nlon=3600)
- TLAT/TLONG가 2D 배열 → `sel(lat=..., lon=...)` 불가, 인덱스 기반 접근 필요
- 033001-033712 / 033801-033912 파일이 분리되어 있으나 시간이 연속되므로 `open_mfdataset`이 자동 병합
- POP 육지 격자는 `_FillValue`로 마스크 → xarray가 자동으로 NaN 처리

## 0. 파일 구조 탐색

In [24]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR   = Path("../data/raw/cesm_hr/sst")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 단일 파일로 구조 확인
sample = sorted(DATA_DIR.glob("*.nc"))[0]
ds0 = xr.open_dataset(sample)
print(ds0)
print("\n--- 좌표 ---")
for c in ds0.coords:
    print(f"  {c}: dims={ds0.coords[c].dims}, shape={ds0.coords[c].shape}")
print("\n--- SST 변수 ---")
print(ds0["SST"])
ds0.close()

<xarray.Dataset> Size: 6GB
Dimensions:             (nlat: 2400, nlon: 3600, time: 120, z_t: 1, z_w: 62,
                         d2: 2, z_t_150m: 15, z_w_bot: 62, z_w_top: 62)
Coordinates:
    TLAT                (nlat, nlon) float64 69MB ...
    TLONG               (nlat, nlon) float64 69MB ...
    ULAT                (nlat, nlon) float64 69MB ...
    ULONG               (nlat, nlon) float64 69MB ...
  * time                (time) object 960B 0250-02-01 00:00:00 ... 0260-01-01...
  * z_t                 (z_t) float32 4B 500.0
  * z_t_150m            (z_t_150m) float32 60B 500.0 1.5e+03 ... 1.45e+04
  * z_w                 (z_w) float32 248B 0.0 1e+03 2e+03 ... 5.5e+05 5.75e+05
  * z_w_bot             (z_w_bot) float32 248B 1e+03 2e+03 ... 5.75e+05 6e+05
  * z_w_top             (z_w_top) float32 248B 0.0 1e+03 ... 5.5e+05 5.75e+05
Dimensions without coordinates: nlat, nlon, d2
Data variables: (12/51)
    ANGLE               (nlat, nlon) float64 69MB ...
    ANGLET              (nlat, n

C:\Users\dnjst\AppData\Local\Temp\ipykernel_9948\1170513661.py:14: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds0 = xr.open_dataset(sample)


## 1. 인덱스 바운딩박스 계산 + 데이터 로드

In [25]:
DOMAINS = {
    "a": {"lat": (10.0, 55.0), "lon": (105.0, 180.0)},
    "b": {"lat": (15.0, 45.0), "lon": (120.0, 180.0)},
    "c": {"lat": (20.0, 35.0), "lon": (125.0, 150.0)},
}

ALL_FILES = sorted(DATA_DIR.glob("*.nc"))
print(f"파일 수: {len(ALL_FILES)}")
for f in ALL_FILES:
    print(f"  {f.name}")

# 전체 TLAT/TLONG는 모든 파일에서 동일 → 참조 파일에서 1회만 로드
ref_ds     = xr.open_dataset(ALL_FILES[0])
TLAT_FULL  = ref_ds["TLAT"].values   # (2400, 3600)
TLONG_FULL = ref_ds["TLONG"].values
ref_ds.close()
print(f"\nTLAT  range: {TLAT_FULL.min():.2f} ~ {TLAT_FULL.max():.2f}")
print(f"TLONG range: {TLONG_FULL.min():.2f} ~ {TLONG_FULL.max():.2f}")

C:\Users\dnjst\AppData\Local\Temp\ipykernel_9948\1604946988.py:13: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ref_ds     = xr.open_dataset(ALL_FILES[0])


파일 수: 11
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.025001-025912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.026001-026912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.027001-027912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.028001-028912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.029001-029912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.030001-030912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.031001-031912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.032001-032912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.033001-033712.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.033801-033912.nc
  B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02.pop.h.SST.034001-034912.nc

TLAT  range: -78.47 ~ 89.97
TLONG range: -1.00 ~ 360.00


In [26]:
def bbox_indices(tlat, tlong, lat_min, lat_max, lon_min, lon_max):
    """2D 격자에서 lat/lon 범위에 해당하는 (nlat_slice, nlon_slice) 반환."""
    mask = (
        (tlat  >= lat_min) & (tlat  <= lat_max) &
        (tlong >= lon_min) & (tlong <= lon_max)
    )
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return slice(int(rows[0]), int(rows[-1]) + 1), slice(int(cols[0]), int(cols[-1]) + 1)


# 도메인 a 바운딩박스
nlat_a, nlon_a = bbox_indices(TLAT_FULL, TLONG_FULL, *DOMAINS["a"]["lat"], *DOMAINS["a"]["lon"])
TLAT_A  = TLAT_FULL[nlat_a, nlon_a]
TLONG_A = TLONG_FULL[nlat_a, nlon_a]
print(f"도메인 a 바운딩박스: {TLAT_A.shape}")

# ── preprocess ─────────────────────────────────────────────────────────
# 파일에 따라 SST dims = (time, z_t, nlat, nlon) 또는 (time, nlat, nlon)
# → z_t가 있으면 isel로 제거, 없으면 그대로
def preprocess(ds):
    sst = ds["SST"]
    if "z_t" in sst.dims:
        sst = sst.isel(z_t=0, drop=True)
    return sst.isel(nlat=nlat_a, nlon=nlon_a).to_dataset(name="SST")

# combine="nested" + concat_dim="time" : 시간축 순서만 맞추면 되므로 가장 안전
ds_all = xr.open_mfdataset(
    ALL_FILES,
    combine="nested",
    concat_dim="time",
    preprocess=preprocess,
    parallel=False,
)
print(f"ds_all: {dict(ds_all['SST'].sizes)}")
print(f"시간   : {str(ds_all.time.values[0])[:10]} ~ {str(ds_all.time.values[-1])[:10]}  ({len(ds_all.time)}개월)")

도메인 a 바운딩박스: (517, 750)


C:\Users\dnjst\AppData\Local\Temp\ipykernel_9948\1635397405.py:28: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_all = xr.open_mfdataset(
C:\Users\dnjst\AppData\Local\Temp\ipykernel_9948\1635397405.py:28: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default wa

ds_all: {'time': 1200, 'nlat': 517, 'nlon': 750}
시간   : 0250-02-01 ~ 0350-01-01  (1200개월)


## 2. 도메인별 서브셋 + 마스킹

In [27]:
SST_DOMAINS = {}

for k, d in DOMAINS.items():
    lat_min, lat_max = d["lat"]
    lon_min, lon_max = d["lon"]

    # 도메인 a 서브그리드 안에서 이 도메인의 인덱스 범위
    nlat_k, nlon_k = bbox_indices(TLAT_A, TLONG_A, lat_min, lat_max, lon_min, lon_max)
    tlat_k  = TLAT_A[nlat_k, nlon_k]
    tlong_k = TLONG_A[nlat_k, nlon_k]

    # 정밀 위경도 마스크
    geo_mask = (
        (tlat_k  >= lat_min) & (tlat_k  <= lat_max) &
        (tlong_k >= lon_min) & (tlong_k <= lon_max)
    )

    sst_k = (
        ds_all["SST"]
        .isel(nlat=nlat_k, nlon=nlon_k)
        .where(geo_mask)
    )
    sst_k = sst_k.assign_coords(
        TLAT=(["nlat", "nlon"], tlat_k),
        TLONG=(["nlat", "nlon"], tlong_k),
    )

    SST_DOMAINS[k] = sst_k

    # ── 실제 유효 격자(마스크 안쪽) 기준으로 범위 출력 ────────────────
    valid_lat  = tlat_k[geo_mask]
    valid_lon  = tlong_k[geo_mask]
    print(f"domain {k}: array {dict(sst_k.sizes)}")
    print(f"  목표  lat {lat_min}–{lat_max}N, lon {lon_min}–{lon_max}E")
    print(f"  유효격자 lat {valid_lat.min():.2f}–{valid_lat.max():.2f}, "
          f"lon {valid_lon.min():.2f}–{valid_lon.max():.2f}  "
          f"({geo_mask.sum():,}개 / {geo_mask.size:,}개)")

domain a: array {'time': 1200, 'nlat': 517, 'nlon': 750}
  목표  lat 10.0–55.0N, lon 105.0–180.0E
  유효격자 lat 10.10–55.00, lon 105.05–180.00  (294,183개 / 387,750개)
domain b: array {'time': 1200, 'nlat': 331, 'nlon': 600}
  목표  lat 15.0–45.0N, lon 120.0–180.0E
  유효격자 lat 15.07–45.00, lon 120.00–180.00  (190,817개 / 198,600개)
domain c: array {'time': 1200, 'nlat': 166, 'nlon': 250}
  목표  lat 20.0–35.0N, lon 125.0–150.0E
  유효격자 lat 20.03–35.00, lon 125.00–149.95  (40,607개 / 41,500개)


## 3. 데이터 검증 + 시각화

In [28]:
# ── 1) 실체화 ──────────────────────────────────────────────────────────
print("compute 중 (SST 실체화)...")
SST_COMPUTED = {k: SST_DOMAINS[k].compute() for k in DOMAINS}
ds_all.close()
print("완료\n")

# ── 2) 도메인 마스킹 검증 ──────────────────────────────────────────────
print("=== 도메인 마스킹 검증 ===")
for k, d in DOMAINS.items():
    lat_min, lat_max = d["lat"]
    lon_min, lon_max = d["lon"]

    sst_snap = SST_COMPUTED[k].isel(time=0).values
    tlat_k   = SST_COMPUTED[k].coords["TLAT"].values
    tlong_k  = SST_COMPUTED[k].coords["TLONG"].values

    outside = (
        (tlat_k  < lat_min) | (tlat_k  > lat_max) |
        (tlong_k < lon_min) | (tlong_k > lon_max)
    )
    n_outside_valid = int(np.sum(~np.isnan(sst_snap[outside])))
    n_total_valid   = int(np.sum(~np.isnan(sst_snap)))

    print(f"\ndomain {k}:")
    print(f"  목표 lat {lat_min}–{lat_max}, lon {lon_min}–{lon_max}")
    print(f"  전체 유효 격자: {n_total_valid:,}")
    print(f"  도메인 외부 유효 격자: {n_outside_valid}  ← 0이어야 정상")

# ── 3) 저장 ───────────────────────────────────────────────────────────
print("\n=== NetCDF 저장 ===")
for k, d in DOMAINS.items():
    out_path = OUTPUT_DIR / f"cesm_hr_sst_domain{k}.nc"
    if out_path.exists():
        out_path.unlink()

    out_ds = SST_COMPUTED[k].to_dataset(name="SST")
    out_ds.attrs.update({
        "title"  : f"CESM-HR monthly SST — domain {k}",
        "source" : "B.E.13.B1850C5.ne120_t12.sehires38.003.sunway_02 POP monthly",
        "domain" : f"lat {d['lat'][0]}–{d['lat'][1]}N, lon {d['lon'][0]}–{d['lon'][1]}E",
        "grid"   : "POP 1/10deg tripolar — TLAT/TLONG are 2D coordinate arrays",
    })
    out_ds.to_netcdf(
        out_path,
        encoding={"SST": {"zlib": True, "complevel": 4, "dtype": "float32"}},
    )
    print(f"저장 완료: {out_path.name}  {dict(SST_COMPUTED[k].sizes)}")

# ── 4) 저장 후 검증 ────────────────────────────────────────────────────
print("\n=== 저장 파일 확인 ===")
for k in DOMAINS:
    path = OUTPUT_DIR / f"cesm_hr_sst_domain{k}.nc"
    ds_chk = xr.open_dataset(path)
    nan_pct = float(ds_chk["SST"].isnull().mean().values) * 100
    print(f"domain {k}: {dict(ds_chk['SST'].sizes)}  NaN {nan_pct:.1f}%  "
          f"time {str(ds_chk.time.values[0])[:10]} ~ {str(ds_chk.time.values[-1])[:10]}")
    ds_chk.close()

compute 중 (SST 실체화)...
완료

=== 도메인 마스킹 검증 ===

domain a:
  목표 lat 10.0–55.0, lon 105.0–180.0
  전체 유효 격자: 272,026
  도메인 외부 유효 격자: 0  ← 0이어야 정상

domain b:
  목표 lat 15.0–45.0, lon 120.0–180.0
  전체 유효 격자: 180,083
  도메인 외부 유효 격자: 0  ← 0이어야 정상

domain c:
  목표 lat 20.0–35.0, lon 125.0–150.0
  전체 유효 격자: 39,211
  도메인 외부 유효 격자: 0  ← 0이어야 정상

=== NetCDF 저장 ===
저장 완료: cesm_hr_sst_domaina.nc  {'time': 1200, 'nlat': 517, 'nlon': 750}
저장 완료: cesm_hr_sst_domainb.nc  {'time': 1200, 'nlat': 331, 'nlon': 600}
저장 완료: cesm_hr_sst_domainc.nc  {'time': 1200, 'nlat': 166, 'nlon': 250}

=== 저장 파일 확인 ===
domain a: {'time': 1200, 'nlat': 517, 'nlon': 750}  NaN 29.8%  time 0250-02-01 ~ 0350-01-01
domain b: {'time': 1200, 'nlat': 331, 'nlon': 600}  NaN 9.3%  time 0250-02-01 ~ 0350-01-01
domain c: {'time': 1200, 'nlat': 166, 'nlon': 250}  NaN 5.5%  time 0250-02-01 ~ 0350-01-01
